In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
!nvidia-smi

Tue May 12 23:20:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
%%writefile vector_add1.cu
#include <iostream>
#include <vector>
#include <ctime>
using namespace std;


__global__ void vector_add_gpu(int* a, int* b, int* c, int n){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if(i < n) c[i] = a[i] + b[i];
}

// Runs on CPU — sequential loop
void vector_add_cpu(vector<int>& a, vector<int>& b, vector<int>& c, int n){
    for(int i = 0; i < n; i++) c[i] = a[i] + b[i];
}

int main(){
    int n = 1000000;

    // Host (CPU) memory
    vector<int> h_a(n), h_b(n), h_c_cpu(n), h_c_gpu(n);
    for(int i = 0; i < n; i++){ h_a[i] = i; h_b[i] = i * 2; }

    // Sequential
    clock_t cpu_start = clock();
    vector_add_cpu(h_a, h_b, h_c_cpu, n);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000;
    cout << "Sequential: " << cpu_time << " ms" << endl;

    // Parallel
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, n * sizeof(int));          // allocate GPU memory
    cudaMalloc(&d_b, n * sizeof(int));
    cudaMalloc(&d_c, n * sizeof(int));

    cudaMemcpy(d_a, h_a.data(), n * sizeof(int), cudaMemcpyHostToDevice);  // CPU → GPU
    cudaMemcpy(d_b, h_b.data(), n * sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    vector_add_gpu<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);  // launch

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);
    cout << "Parallel (GPU): " << gpu_time << " ms" << endl;

    cudaMemcpy(h_c_gpu.data(), d_c, n * sizeof(int), cudaMemcpyDeviceToHost);  // GPU → CPU

    // Verify
    bool correct = true;
    for(int i = 0; i < n; i++) if(h_c_cpu[i] != h_c_gpu[i]){ correct = false; break; }
    cout << "Correct: " << (correct ? "YES" : "NO") << endl;
    cout << "Speedup: " << cpu_time / gpu_time << "x" << endl;

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing vector_add1.cu


In [16]:
!nvcc vector_add1.cu -o vector_add1 && ./vector_add1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Sequential: 7.281 ms
Parallel (GPU): 0.159776 ms
Correct: YES
Speedup: 45.57x
